# Retenção das soluções durante a recomposição do C-NBI

Este notebook apresenta as medianas de retenção em três etapas da recomposição do C-NBI: soluções válidas, soluções únicas após a remoção de duplicatas e soluções globalmente não dominadas. Os três painéis correspondem a $m=4$, $m=6$ e $m=12$; em cada painel, as linhas representam correlação baixa, média e alta.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'cnbi_recomposition'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_WIDTH_CM = 16.0
CM_TO_INCH = 1 / 2.54

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.5,
    'axes.titlesize': 10.5,
    'axes.labelsize': 9.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 8.0,
    'legend.fontsize': 8.5,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})
print('Saída:', OUT_DIR)

In [ ]:
records = [
    ('m4_low', 4, 'low', 100.00, 93.61, 74.87),
    ('m4_medium', 4, 'medium', 100.00, 88.48, 87.88),
    ('m4_high', 4, 'high', 100.00, 93.24, 49.22),
    ('m6_low', 6, 'low', 100.00, 93.91, 86.71),
    ('m6_medium', 6, 'medium', 100.00, 93.92, 45.50),
    ('m6_high', 6, 'high', 100.00, 91.98, 38.19),
    ('m12_low', 12, 'low', 100.00, 93.95, 74.13),
    ('m12_medium', 12, 'medium', 100.00, 93.94, 42.71),
    ('m12_high', 12, 'high', 100.00, 93.34, 27.12),
]
columns = ['scenario', 'm', 'correlation', 'valid', 'unique', 'nondominated']
retention = pd.DataFrame(records, columns=columns)

assert len(retention) == 9
assert retention['valid'].eq(100.0).all()
stage_values = retention[['valid', 'unique', 'nondominated']]
assert stage_values.ge(0).all().all() and stage_values.le(100).all().all()
assert (retention['valid'] >= retention['unique']).all()
assert (retention['unique'] >= retention['nondominated']).all()

retention['duplicate_reduction'] = retention['valid'] - retention['unique']
retention['dominance_reduction'] = retention['unique'] - retention['nondominated']
data_path = OUT_DIR / 'median_retention_cnbi.csv'
retention.to_csv(data_path, index=False)
retention

In [ ]:
STAGES = ['valid', 'unique', 'nondominated']
STAGE_LABELS = ['Válidas', 'Únicas', 'Não\ndominadas\nglobais']
CORRELATIONS = ['low', 'medium', 'high']
CORRELATION_LABELS = {'low': 'baixa', 'medium': 'média', 'high': 'alta'}
COLORS = {'low': '#2166AC', 'medium': '#E08214', 'high': '#B2182B'}
MARKERS = {'low': 'o', 'medium': 's', 'high': '^'}
LINESTYLES = {'low': '-', 'medium': '--', 'high': ':'}
x = np.arange(len(STAGES))

fig, axes = plt.subplots(
    1, 3, sharey=True,
    figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, 9.0 * CM_TO_INCH),
)
fig.subplots_adjust(left=0.09, right=0.985, top=0.89, bottom=0.31, wspace=0.16)

for ax, m in zip(axes, [4, 6, 12]):
    panel = retention.loc[retention['m'].eq(m)].set_index('correlation')
    for correlation in CORRELATIONS:
        values = panel.loc[correlation, STAGES].to_numpy(dtype=float)
        ax.plot(
            x, values,
            color=COLORS[correlation],
            marker=MARKERS[correlation],
            linestyle=LINESTYLES[correlation],
            linewidth=1.55, markersize=5.2,
            label=CORRELATION_LABELS[correlation],
        )
        final_label = f'{values[-1]:.1f}%'.replace('.', ',')
        vertical_offset = 7 if correlation in ('low', 'medium') else -8
        vertical_alignment = 'bottom' if vertical_offset > 0 else 'top'
        ax.annotate(
            final_label, (x[-1], values[-1]), xytext=(0, vertical_offset),
            textcoords='offset points', ha='center', va=vertical_alignment,
            fontsize=6.7, color=COLORS[correlation],
        )

    ax.set_title(rf'$m={m}$', pad=6)
    ax.set_xticks(x, STAGE_LABELS)
    ax.set_xlim(-0.08, 2.35)
    ax.set_ylim(0, 104)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.tick_params(axis='x', labelsize=7.0)
    ax.grid(axis='y', alpha=0.20, linewidth=0.55)
    ax.axhline(100, color='0.45', linewidth=0.65, linestyle='--', zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_color('0.35')
        spine.set_linewidth(0.7)

axes[0].set_ylabel('Retenção mediana (%)')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, title='Correlação',
    loc='lower center', bbox_to_anchor=(0.5, 0.025),
    ncol=3, frameon=False,
)

figure_png = OUT_DIR / 'retencao_solucoes_recomposicao_cnbi.png'
figure_pdf = OUT_DIR / 'retencao_solucoes_recomposicao_cnbi.pdf'
fig.savefig(figure_png, dpi=300)
fig.savefig(figure_pdf, dpi=300)
plt.close(fig)

metadata = {
    'measure': 'median retention percentage',
    'panels': [4, 6, 12],
    'correlation_levels': CORRELATIONS,
    'stages': STAGES,
    'publication_width_cm': FIGURE_WIDTH_CM,
    'main_reading': 'duplicate removal is small and relatively uniform; the principal loss occurs during global nondominance filtering',
}
metadata_path = OUT_DIR / 'retencao_solucoes_recomposicao_cnbi_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

for artifact in [data_path, figure_png, figure_pdf, metadata_path]:
    assert artifact.exists() and artifact.stat().st_size > 0
print('Redução por duplicatas:', retention['duplicate_reduction'].min(), 'a', retention['duplicate_reduction'].max())
print('Redução por dominância:', retention['dominance_reduction'].min(), 'a', retention['dominance_reduction'].max())
print('Figura:', figure_png.relative_to(ROOT).as_posix())

## Leitura da figura

A passagem de soluções válidas para soluções únicas produz uma redução pequena e relativamente estável entre os nove cenários. A maior perda ocorre na filtragem global de dominância. Esse efeito é particularmente forte nos cenários de correlação média e alta, sobretudo à medida que o número de objetivos aumenta.